# 01 — Recopilación de datos

Este notebook documenta el proceso de recopilación de datos para el Índice Chocorramo.

## Fuentes principales
- **Colombia salario mínimo:** DANE / Función Pública (Decretos)
- **Colombia IPC:** DANE
- **Australia salario mínimo:** Fair Work Commission
- **Australia CPI:** ABS
- **Precios de productos:** estimaciones (ver `docs/source_quality.md`)

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
RAW_DIR = ROOT / 'data' / 'raw'
print('Raw data directory:', RAW_DIR)
print('\nFiles available:')
for f in sorted(RAW_DIR.glob('*.csv')):
    print(f'  {f.name}')

## Colombia — Salario Mínimo

In [ ]:
co_wage = pd.read_csv(RAW_DIR / 'colombia_min_wage.csv')
co_wage_monthly = co_wage[co_wage['variable'] == 'min_wage_monthly'].copy()
co_wage_monthly[['year', 'value', 'confidence', 'needs_verification']].sort_values('year')

## Colombia — Precios Chocorramo

> ⚠️ **IMPORTANTE:** Los precios del Chocorramo son estimaciones. Ver `docs/source_quality.md`.

In [ ]:
co_prices = pd.read_csv(RAW_DIR / 'colombia_chocorramo_prices.csv')
print('Confidence levels:', co_prices['confidence'].unique())
print('All need verification:', co_prices['needs_verification'].all())
co_prices[['year', 'value', 'source_type', 'confidence', 'notes']].to_string()

## Colombia — IPC (DANE)

In [ ]:
co_cpi = pd.read_csv(RAW_DIR / 'colombia_cpi.csv')
co_annual = co_cpi[co_cpi['variable'] == 'annual_inflation_pct'].copy()
co_cumulative = co_cpi[co_cpi['variable'] == 'cpi_base2016'].copy()

merged = co_annual.merge(co_cumulative, on='year', suffixes=('_infl', '_cpi'))
merged[['year', 'value_infl', 'value_cpi', 'needs_verification_infl']].rename(
    columns={'value_infl': 'inflacion_anual_%', 'value_cpi': 'IPC_base2016=100', 'needs_verification_infl': 'estimado'}
).set_index('year')

## Australia — Salario Mínimo (Fair Work Commission)

In [ ]:
au_wage = pd.read_csv(RAW_DIR / 'australia_min_wage.csv')
au_weekly = au_wage[au_wage['variable'] == 'min_wage_weekly']
au_weekly[['year', 'value', 'confidence', 'needs_verification']].sort_values('year')

## Australia — Precios Meat Pie

> ⚠️ **IMPORTANTE:** El precio del meat pie es un precio representativo, no un dato oficial.

In [ ]:
au_prices = pd.read_csv(RAW_DIR / 'australia_meat_pie_prices.csv')
au_prices[['year', 'value', 'source_type', 'confidence']].sort_values('year')

## Resumen de estado de verificación

In [ ]:
all_dfs = {
    'colombia_wages': co_wage,
    'colombia_prices': co_prices,
    'colombia_cpi': co_cpi,
    'australia_wages': au_wage,
    'australia_prices': au_prices,
}

print('Datos que requieren verificación:')
for name, df in all_dfs.items():
    needs_v = df[df['needs_verification'].astype(str).str.lower() == 'true']
    if not needs_v.empty:
        print(f'\n  {name} ({len(needs_v)} filas):')
        for _, row in needs_v.iterrows():
            print(f'    [{row["year"]}] {row["variable"]} = {row["value"]} — {row["source_type"]}')